In [ ]:
import numpy as np

import matplotlib
matplotlib.use('TkAgg') 
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

from scipy import signal
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import gaussian

import time
import os
import sys
import socket

In [11]:
def stft(fs, data, window, hop):
    
    SFT = ShortTimeFFT(window, hop=hop, fs=fs, mfft=200, scale_to='magnitude')
    Sx = SFT.stft(data)  # perform the STFT
    return Sx, SFT

In [12]:
def open_spectrograph():


    # ==========================================
    # 1. PARAMETRY SYSTEMOWE I FIZYCZNE
    # ==========================================
   
    FS = 44100            
    CHUNK_SIZE = 2048     
    NPERSEG = 512         
    HISTORY_LEN = 150     

    BIND_IP = "127.0.0.1"
    PORT = 5005



    # Wyliczanie parametrów czasu dla osi odciętych
    TIME_PER_FRAME = CHUNK_SIZE / FS
    TOTAL_HISTORY_TIME = HISTORY_LEN * TIME_PER_FRAME

    # Wyliczanie liczby prążków dla dyskretnej transformaty Fouriera
    freq_bins = NPERSEG // 2 + 1
    waterfall = np.full((freq_bins, HISTORY_LEN), -100.0)

    # ==========================================
    # 2. INICJALIZACJA INTERFEJSU GRAFICZNEGO
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 7))

    # Rezerwacja dolnej przestrzeni okna na suwaki
    plt.subplots_adjust(bottom=0.2) 

    # Inicjalizacja macierzy pikseli (Początkowy widok od ujemnego czasu historii do zera)
    cax = ax.imshow(waterfall, aspect='auto', origin='lower', cmap='inferno', 
                    vmin=-100, vmax=0, extent=[-TOTAL_HISTORY_TIME, 0, 0, FS/2])

    ax.set_title("Real-Time STFT - Czas Bezwzględny (Płynna Oś)")
    ax.set_ylabel("Częstotliwość [Hz]")
    ax.set_xlabel("Czas bezwzględny [s]")
    fig.colorbar(cax, label="Amplituda [dB]")

    # --- KONFIGURACJA WIDŻETÓW STERUJĄCYCH ---
    ax_slider_min = plt.axes([0.15, 0.1, 0.65, 0.03])
    ax_slider_max = plt.axes([0.15, 0.05, 0.65, 0.03])

    slider_min = Slider(ax_slider_min, 'Min dB', -150, 50, valinit=-100)
    slider_max = Slider(ax_slider_max, 'Max dB', -100, 100, valinit=0)

    def on_slider_change(val):
        """Przerwanie aktualizujące progi obcinania palety kolorów."""
        cax.set_clim(vmin=slider_min.val, vmax=slider_max.val)
        fig.canvas.draw_idle()

    slider_min.on_changed(on_slider_change)
    slider_max.on_changed(on_slider_change)

    plt.show(block=False)


    import sys
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:
        sock.bind((BIND_IP, PORT))
        sock.setblocking(False)
        print(f"[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na {BIND_IP}:{PORT}")
    except OSError as e:
        print(f"[BŁĄD KRYTYCZNY] Nie można zbindować gniazda. Port może być zablokowany: {e}")
        sys.exit(1)
    # ==========================================
    # 4. PĘTLA WYKONAWCZA CZASU RZECZYWISTEGO
    # ==========================================
    total_frames = 0
    last_draw_time = time.time()
    DRAW_INTERVAL = 1.0 / 30.0  # Limit odświeżania GUI do 30 FPS

    try:
        while True:
            processed_any = False
            
            # Drenaż bufora jądra - przetwarzanie wszystkich napływających ramek
            while True:
                try:
                    data = sock.recv(CHUNK_SIZE * 4) 
                    
                    # Dekodowanie i obliczanie STFT dla KAŻDEJ ramki
                    chunk = np.frombuffer(data, dtype=np.float32)
                    f_bins, t_bins, Zxx = signal.stft(chunk, fs=FS, nperseg=NPERSEG)
                    slice_idx = Zxx.shape[1] // 2
                    magnitude_db = 20 * np.log10(np.abs(Zxx[:, slice_idx]) + 1e-9) 
                    
                    # Aktualizacja macierzy pikseli w pamięci
                    waterfall = np.roll(waterfall, shift=-1, axis=1)
                    waterfall[:, -1] = magnitude_db
                    
                    total_frames += 1
                    processed_any = True
                    
                except BlockingIOError:
                    break # Brak danych w buforze IPC
                    
            # --- ZARZĄDZANIE WIZUALIZACJĄ (DECIMATION) ---
            if processed_any:
                current_time = time.time()
                
                # Odrysowanie tylko jeśli minął interwał czasu (odciążenie procesora)
                if (current_time - last_draw_time) >= DRAW_INTERVAL:
                    cax.set_data(waterfall)
                    
                    # Logika przesuwania osi czasu na podstawie WSZYSTKICH przetworzonych ramek
                    sim_time = total_frames * TIME_PER_FRAME
                    start_time = sim_time - TOTAL_HISTORY_TIME
                    
                    cax.set_extent([start_time, sim_time, 0, FS/2])
                    ax.set_xlim(start_time, sim_time)
                    
                    fig.canvas.draw_idle()
                    last_draw_time = current_time
            
            # Asynchroniczne przetworzenie zdarzeń (suwaki) niezależnie od odrysowania
            fig.canvas.flush_events()
            
            # Zabezpieczenie przed 100% użyciem CPU w przypadku pustego bufora IPC
            if not processed_any:
                time.sleep(0.005)

    except KeyboardInterrupt:
        print("\n[SYSTEM] Otrzymano sygnał przerwania. Rozpoczynanie zwalniania zasobów...")
    # ==========================================
    # 5. PROCEDURA ZWALNIANIA ZASOBÓW
    # ==========================================
    sock.close()
    
    print("[SYSTEM] Zakończono strumieniowanie. Deskryptor pliku został usunięty.")

In [ ]:
# when quitting matpliotlib close sockets and whatnot

In [13]:
open_spectrograph()

[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na 127.0.0.1:5005

[SYSTEM] Otrzymano sygnał przerwania. Rozpoczynanie zwalniania zasobów...
[SYSTEM] Zakończono strumieniowanie. Deskryptor pliku został usunięty.


In [10]:
%%bash --bg
# Zapis PID powłoki
echo $$ > /tmp/mic_tx.pid

# Aktywacja środowiska i przejście do katalogu roboczego
source ../bin/activate
cd src
python -u net_tx_mic.py

In [11]:
# Odczyt PID, wysłanie sygnału SIGTERM i usunięcie wskaźników
!kill -15 $(cat /tmp/mic_tx.pid) && rm /tmp/mic_tx.pid && echo "[SYSTEM] Proces zakończony. Logi pozostały w /tmp/mic_tx.log"


[SYSTEM] Proces zakończony. Logi pozostały w /tmp/mic_tx.log
